# Next-item prediction over Semantic IDs (RecGPT)
Flatten user histories into Semantic ID tokens and train a tiny GPT to generate the next item.


In [ ]:
import sys; sys.path.append('..')
import numpy as np, torch, matplotlib.pyplot as plt
from config import cfg
from tokenizer import build_tokenizer
from model import GPTConfig, RecGPT
from rqvae import disambiguate
from eval import beam_search_items
from common import get_device, seed_everything
seed_everything(0); device = get_device()

In [ ]:
# synthetic catalog: an item's FIRST code = its cluster, so cluster-mates share a prefix
rng = np.random.default_rng(0)
n_items, C = 300, 6
cluster = rng.integers(0, C, size=n_items)
codes = np.stack([cluster,
                  rng.integers(0, 16, size=n_items),
                  rng.integers(0, 16, size=n_items)], axis=1)
codes = disambiguate(codes)            # append a uniqueness digit
tok = build_tokenizer(codes.tolist())
tok.vocab_size, tok.D

In [ ]:
# each user browses within one cluster -> the next item is predictable from history
seqs = []
for _ in range(800):
    c = rng.integers(0, C)
    members = np.where(cluster == c)[0]
    seqs.append([int(i) for i in rng.choice(members, size=6)])
block_size = cfg.max_seq_len * tok.D + 1

In [ ]:
# flatten to tokens, right-pad, build the model
X, Y = [], []
for s in seqs:
    t = tok.encode_sequence(s[:-1])[:block_size+1]
    xi, yi = t[:-1], t[1:]; pad = block_size - len(xi)
    X.append(xi + [tok.PAD]*pad); Y.append(yi + [tok.PAD]*pad)
X = torch.tensor(X, device=device); Y = torch.tensor(Y, device=device)
gptcfg = GPTConfig(vocab_size=tok.vocab_size, block_size=block_size,
                   n_layer=2, n_head=2, n_embd=64, dropout=0.0, pad_token=tok.PAD)
model = RecGPT(gptcfg).to(device)
opt = model.configure_optimizers(0.1, 3e-3, device)

In [ ]:
hist = []
for epoch in range(80):
    perm = torch.randperm(X.shape[0], device=device)
    _, loss = model(X[perm], Y[perm])
    loss.backward(); opt.step(); opt.zero_grad(); hist.append(loss.item())
plt.plot(hist); plt.xlabel('epoch'); plt.ylabel('loss'); plt.show()

In [ ]:
# generate the next item by constrained beam search over Semantic ID digits
model.eval()
history = seqs[0][:-1]
preds = beam_search_items(model, tok, history, device, beam_size=20, max_items_out=5)
print('history clusters :', [int(cluster[i]) for i in history])
print('predicted clusters:', [int(cluster[i]) for i in preds])
print('predicted items   :', preds)